In [1]:
import crypten
import torch


crypten.init()
torch.set_num_threads(1)

In [2]:
#torch.set_printoptions(sci_mode=False)
import crypten.mpc as mpc
import crypten.communicator as comm


@mpc.run_multiprocess(world_size=2)
def norm_multiproc(c):
    rank = comm.get().get_rank()
    c_enc = crypten.cryptensor(c)
    l2_norm = c_enc.norm(p=2,dim=1,keepdim=True)
    crypten.print(f"Rank {rank} \n\t norm : {l2_norm.get_plain_text()}")
    clip_threshold = 1.0
    clip = l2_norm.gt(clip_threshold)
    inv_clip= 1/clip_threshold
    c_enc = (c_enc / (l2_norm*inv_clip) - c_enc) * clip + c_enc

    crypten.print(f"Rank: {rank}\n\t Clipped gradient: {c_enc.get_plain_text()}")#, in_order=True)


c = torch.rand(100, 3, 28, 28)
max_norm = 1.0
# Compute the L2 norm along dim=1
norm = torch.linalg.norm(c, dim=1,keepdim=True)
print(norm)
# Determine the scaling factor
scaling_factor = torch.clamp(norm / max_norm, min=1)
print(scaling_factor)
# Multiply the tensor by the scaling factor
c_clipped = c / scaling_factor
print("Original tensor:", c)
print("Clipped tensor:", c_clipped)

norm_multiproc(c)

tensor([[[[1.0141, 0.9999, 0.8815,  ..., 0.6786, 0.5239, 1.1540],
          [0.8668, 1.4116, 0.8856,  ..., 1.0162, 0.9872, 1.0768],
          [1.4698, 1.1685, 0.7086,  ..., 1.2419, 0.7389, 1.2804],
          ...,
          [1.3938, 0.2028, 0.9585,  ..., 1.0993, 0.8012, 1.6375],
          [1.0204, 1.2712, 0.7798,  ..., 0.8584, 1.0391, 0.5498],
          [0.8839, 0.9743, 0.7748,  ..., 1.0150, 1.1083, 1.1997]]],


        [[[0.8606, 1.2005, 0.9411,  ..., 0.8338, 1.0800, 0.7752],
          [0.8770, 0.7366, 0.7165,  ..., 0.5938, 0.8884, 0.4842],
          [1.1165, 0.8554, 1.1224,  ..., 0.2985, 0.6209, 0.9325],
          ...,
          [1.0936, 1.2749, 0.8505,  ..., 1.4491, 0.4342, 0.7283],
          [0.6928, 0.7484, 0.5181,  ..., 1.1646, 1.1649, 0.3262],
          [0.9717, 0.5271, 0.9355,  ..., 1.1198, 0.8203, 1.2633]]],


        [[[0.9753, 0.5066, 0.9269,  ..., 1.4014, 1.4128, 1.0232],
          [0.9753, 0.9354, 1.3706,  ..., 1.2442, 0.7743, 0.9222],
          [1.1020, 1.0342, 0.8167,  ..

[None, None]

# Test clipping optim
Less multipplications and replacing division with multiplication with the reciprocal

In [3]:
import time

@mpc.run_multiprocess(world_size=2)
def norm_multiproc_v1(c):
    """ Basic formula"""
    c_enc = crypten.cryptensor(c)
    l2_norm = c_enc.norm(p=2,dim=1,keepdim=True)
    clip_threshold = 1.0
    clip = l2_norm.gt(clip_threshold)
    c_enc = c_enc / (l2_norm/clip_threshold)*clip + c_enc*(1-clip)

@mpc.run_multiprocess(world_size=2)
def norm_multiproc_v2(c):
    """ Inverse of clipping precomputed"""
    c_enc = crypten.cryptensor(c)
    l2_norm = c_enc.norm(p=2,dim=1,keepdim=True )
    clip_threshold = 1.0
    clip = l2_norm.gt(clip_threshold)
    inv_clip= 1/clip_threshold
    c_enc =  c_enc / (l2_norm*inv_clip) * clip + c_enc*(1-clip)

@mpc.run_multiprocess(world_size=2)
def norm_multiproc_v3(c):
    """ Only one multiplication and inverse of clipping precomputed"""
    c_enc = crypten.cryptensor(c)
    l2_norm = c_enc.norm(p=2,dim=1,keepdim=True )
    clip_threshold = 1.0
    clip = l2_norm.gt(clip_threshold)
    inv_clip= 1/clip_threshold
    c_enc = (c_enc / (l2_norm*inv_clip) - c_enc) * clip + c_enc

@mpc.run_multiprocess(world_size=2)
def norm_multiproc_v4(c):
    """ Where clause"""
    c_enc = crypten.cryptensor(c)
    l2_norm = c_enc.norm(p=2,dim=1,keepdim=True )
    clip_threshold = 1.0
    inv_clip= 1/clip_threshold
    clipped_l2_norm = l2_norm * inv_clip
    clipped_l2_norm = crypten.where(l2_norm > clip_threshold, clipped_l2_norm, l2_norm)
    c_enc = c_enc / clipped_l2_norm


@mpc.run_multiprocess(world_size=2)
def norm_multiproc_v5(c):
    """ No inverse of clipping """
    c_enc = crypten.cryptensor(c)
    l2_norm = c_enc.norm(p=2,dim=1,keepdim=True )
    clip_threshold = 1.0
    clip = l2_norm.gt(clip_threshold)
    c_enc = ((c_enc * clip / l2_norm) - c_enc) * clip + c_enc

c = torch.rand(100, 3, 28, 28)

start_time =time.time()
for i in range (10):
    norm_multiproc_v1(c)
end_time = time.time()

elapsed_time = (end_time - start_time)/10
print(f"Average execution time v_1: {elapsed_time:.5f} seconds")

start_time =time.time()
for i in range (10):
    norm_multiproc_v2(c)
end_time = time.time()

elapsed_time = (end_time - start_time)/10
print(f"Average execution time v_2: {elapsed_time:.5f} seconds")

start_time =time.time()
for i in range (10):
    norm_multiproc_v3(c)
end_time = time.time()

elapsed_time = (end_time - start_time)/10
print(f"Average execution time v_3: {elapsed_time:.5f} seconds")

start_time =time.time()
for i in range (10):
    norm_multiproc_v4(c)
end_time = time.time()

elapsed_time = (end_time - start_time)/10
print(f"Average execution time v_4: {elapsed_time:.5f} seconds")

start_time =time.time()
for i in range (10):
    norm_multiproc_v5(c)
end_time = time.time()

elapsed_time = (end_time - start_time)/10
print(f"Average execution time v_5: {elapsed_time:.5f} seconds")


Average execution time v_1: 0.95457 seconds
Average execution time v_2: 0.96293 seconds
Average execution time v_3: 0.94999 seconds
Average execution time v_4: 1.00274 seconds
Average execution time v_5: 0.98407 seconds


# Noise test

Each party cannot perform local additions, so each noise needs to be generated and secret shared between the parties before being added

In [4]:
# From https://github.com/google-research/federated/blob/master/distributed_dp/distributed_skellam_query.py#L65 
"""Adds Skellam noise to the sample.

    We use difference of two Poisson random variable with lambda hyperparameter
    that equals 'local_stddev**2/2' that results in a standard deviation
    'local_stddev' for the Skellam noise to be added locally.

    Args:
      local_stddev: The standard deviation of the local Skellam noise.
      record: The record to be processed.

    Returns:
      A record with added noise.
    """
# Use float64 as the stddev could be large after quantization.
#local_stddev = tf.cast(local_stddev, tf.float64)
#poisson_lam = 0.5 * local_stddev * local_stddev

def add_skellam_noise(v, poisson_lam=.1):
    shape = (*v.shape, 2)  # Two draws of Poisson
    print("Shape", shape)
    seed = int(time.time() * 10**6)
    print("Seed", seed)
    torch.manual_seed(seed)
    poissons = torch.poisson(torch.tensor([poisson_lam, poisson_lam]).repeat(*shape[:-1], 1))
    print("Skellam noise",(poissons[..., 0] - poissons[..., 1]).to(v.dtype))
    return v + (poissons[..., 0] - poissons[..., 1]).to(v.dtype)

# Example usage
v = torch.tensor([1.0, 2.0, 3.0])
result = add_skellam_noise(v)
print(result)

Shape (3, 2)
Seed 1688387902113434
Skellam noise tensor([2., 0., 0.])
tensor([3., 2., 3.])


In [5]:
# Share data type is torch.int64
def add_skellam_noise_crypten_tensor(v, poisson_lam=1.0):
    shape = (*v.shape, 2)  # Two draws of Poisson
    print("Shape", shape)
    seed = int(time.time() * 10**6)
    print("Seed", seed)
    torch.manual_seed(seed)
    poissons = torch.poisson(torch.tensor([poisson_lam, poisson_lam]).repeat(*shape[:-1], 1))
    print(poissons)
    print("Skellam noise",(poissons[..., 0] - poissons[..., 1]).to(torch.int64))
    return v + (poissons[..., 0] - poissons[..., 1]).to(torch.int64)

# Example usage
v = torch.tensor([1.0, 2.0, 3.0])
v_enc = crypten.cryptensor(v)
print(v_enc)
result = add_skellam_noise_crypten_tensor(v_enc)
print(result)

MPCTensor(
	_tensor=tensor([ 65536, 131072, 196608])
	plain_text=HIDDEN
	ptype=ptype.arithmetic
)
Shape (3, 2)
Seed 1688387902125704
tensor([[1., 0.],
        [2., 1.],
        [1., 1.]])
Skellam noise tensor([1, 1, 0])
MPCTensor(
	_tensor=tensor([131072, 196608, 196608])
	plain_text=HIDDEN
	ptype=ptype.arithmetic
)


## Test for local noise addition
**NOT WORKING**: only one party adds the noise, the noise has to be secret shared and then added

In [6]:
@mpc.run_multiprocess(world_size=2)
def clip_add_noise(c, poisson_lam=.1, clip_threshold=1.0):
    rank = comm.get().get_rank()
    c_enc = crypten.cryptensor(c)
    l2_norm = c_enc.norm(p=2,dim=1,keepdim=True )
    inv_clip= 1/clip_threshold
    clip = l2_norm.gt(clip_threshold)
    c_enc = (c_enc / (l2_norm*inv_clip) - c_enc) * clip + c_enc
    #crypten.print(f"Clipped vector: {c_enc.get_plain_text()}")
    shape = (*c.shape, 2)  # Two draws of Poisson
    seed = int(time.time() * 10**6)
    #crypten.print(f"Rank {rank}:\n\tseed: {seed}", in_order=True)
    #print(seed)
    torch.manual_seed(seed)
    
    
    poissons = torch.poisson(torch.tensor([poisson_lam, poisson_lam]).repeat(*shape[:-1], 1))
    crypten.print(f"Rank {rank}:\n\tSkellam: {(poissons[..., 0] - poissons[..., 1]).to(c.dtype) }", in_order=True)
    #print((poissons[..., 0] - poissons[..., 1]).to(v.dtype))
    #crypten.print(f"Rank {rank}:\n\tPre-noise: {c_enc}", in_order=True)
   
    c_enc_post = c_enc + (poissons[..., 0] - poissons[..., 1]).to(c.dtype)
    crypten.print(f"Rank {rank}:\n\tPost-noise: {c_enc - c_enc_post}", in_order=True)

    crypten.print(c_enc_post.get_plain_text()-c)


c = torch.rand(2, 1, 2, 2)
print(f"Original c value: {c}")
clip_add_noise(c)



Original c value: tensor([[[[0.8571, 0.4389],
          [0.3018, 0.6509]]],


        [[[0.6452, 0.4129],
          [0.3719, 0.2578]]]])
Rank 0:
	Skellam: tensor([[[[0., 0.],
          [0., 0.]]],


        [[[0., 0.],
          [0., 0.]]]])
Rank 1:
	Skellam: tensor([[[[ 0., -2.],
          [ 0.,  1.]]],


        [[[ 0.,  0.],
          [-1., -1.]]]])
Rank 0:
	Post-noise: MPCTensor(
	_tensor=tensor([[[[0, 0],
          [0, 0]]],


        [[[0, 0],
          [0, 0]]]])
	plain_text=HIDDEN
	ptype=ptype.arithmetic
)
Rank 1:
	Post-noise: MPCTensor(
	_tensor=tensor([[[[0, 0],
          [0, 0]]],


        [[[0, 0],
          [0, 0]]]])
	plain_text=HIDDEN
	ptype=ptype.arithmetic
)
tensor([[[[-3.9935e-06, -1.1921e-05],
          [-2.5034e-06, -1.6093e-06]]],


        [[[-1.2577e-05, -3.9935e-06],
          [-7.3910e-06, -5.6624e-06]]]])


[None, None]

## Test for noise sharing and addition

In [7]:
@mpc.run_multiprocess(world_size=2)
def clip_add_noise(c, poisson_lam=1.0, clip_threshold=1.0):
    rank = comm.get().get_rank()
    c_enc = crypten.cryptensor(c)

    noise = crypten.cryptensor([1], src = rank)
    crypten.print(f"Rank {rank}:\n\tnoise: {noise}", in_order=True)
    crypten.print(f"Rank {rank}:\n\tnoise: {noise.get_plain_text()}", in_order=True)
    c_enc = c_enc + noise

    crypten.print(f"Rank {rank}:\n\tPost-noise: {c_enc}", in_order=True)

    crypten.print(c_enc.get_plain_text())


#c = torch.rand(2, 1, 2, 2)
c = torch.tensor([1,2])
print(f"Original c value: {c}")
clip_add_noise(c)



Original c value: tensor([1, 2])
Rank 0:
	noise: MPCTensor(
	_tensor=tensor([-3605194621576131267])
	plain_text=HIDDEN
	ptype=ptype.arithmetic
)
Rank 1:
	noise: MPCTensor(
	_tensor=tensor([3605194621576262339])
	plain_text=HIDDEN
	ptype=ptype.arithmetic
)
Rank 0:
	noise: tensor([2.])
Rank 1:
	noise: tensor([2.])
Rank 0:
	Post-noise: MPCTensor(
	_tensor=tensor([-8894922116401133654, -5999801319966519060])
	plain_text=HIDDEN
	ptype=ptype.arithmetic
)
Rank 1:
	Post-noise: MPCTensor(
	_tensor=tensor([8894922116401330262, 5999801319966781204])
	plain_text=HIDDEN
	ptype=ptype.arithmetic
)
tensor([3., 4.])


[None, None]

## Skellam noise 

### Toy test for multi poisson sampling

In [8]:
@mpc.run_multiprocess(world_size=2)
def clip_add_noise(c, poisson_lam=1.0, clip_threshold=10.0):
    rank = comm.get().get_rank()
    c_enc = crypten.cryptensor(c)
    l2_norm = c_enc.norm(p=2,dim=1,keepdim=True )
    inv_clip= 1/clip_threshold
    clip = l2_norm.gt(clip_threshold)
    c_enc = (c_enc / (l2_norm*inv_clip) - c_enc) * clip + c_enc
    #crypten.print(f"Clipped vector: {c_enc.get_plain_text()}")
    shape = (*c.shape, 2)  # Two draws of Poisson
    seed = int(time.time() * 10**6)
    #crypten.print(f"Rank {rank}:\n\tseed: {seed}", in_order=True)
    #print(seed)
    torch.manual_seed(seed)
    
    
    poissons = torch.poisson(torch.tensor([poisson_lam, poisson_lam]).repeat(*shape[:-1], 1))
    crypten.print(f"Rank {rank}:\n\tPoissons: {(poissons[..., 0] - poissons[..., 1]).to(torch.int64)}", in_order=True)
    skellam_noise_enc = crypten.cryptensor((poissons[..., 0] - poissons[..., 1]).to(torch.int64), src = rank)
    crypten.print(f"Rank {rank}:\n\tSkellam: {skellam_noise_enc }", in_order=True)
    #print((poissons[..., 0] - poissons[..., 1]).to(v.dtype))
    #crypten.print(f"Rank {rank}:\n\tPre-noise: {c_enc}", in_order=True)
   
    c_enc_post = c_enc + skellam_noise_enc
    crypten.print(f"Rank {rank}:\n\tPost-noise: {c_enc - c_enc_post}", in_order=True)

    crypten.print(c_enc_post.get_plain_text()-c)


c = torch.rand(2, 1, 2, 2)
c = torch.tensor([[1,2]])
print(f"Original c value: {c}")
clip_add_noise(c)



Original c value: tensor([[1, 2]])


Rank 0:
	Poissons: tensor([[-2, -3]])
Rank 1:
	Poissons: tensor([[ 1, -1]])
Rank 0:
	Skellam: MPCTensor(
	_tensor=tensor([[  908212538669185984, -2056132266751653270]])
	plain_text=HIDDEN
	ptype=ptype.arithmetic
)
Rank 1:
	Skellam: MPCTensor(
	_tensor=tensor([[-908212538669251520, 2056132266751391126]])
	plain_text=HIDDEN
	ptype=ptype.arithmetic
)
Rank 0:
	Post-noise: MPCTensor(
	_tensor=tensor([[-908212538669185984, 2056132266751653270]])
	plain_text=HIDDEN
	ptype=ptype.arithmetic
)
Rank 1:
	Post-noise: MPCTensor(
	_tensor=tensor([[  908212538669251520, -2056132266751391126]])
	plain_text=HIDDEN
	ptype=ptype.arithmetic
)
tensor([[-1., -4.]])


[None, None]

In [10]:
@mpc.run_multiprocess(world_size=2)
def clip_add_noise(c, poisson_lam=1.0, clip_threshold=10.0):
    rank = comm.get().get_rank()
    c_enc = crypten.cryptensor(c)
    dim = 1 % len(c_enc.shape)
    if not dim:
        l2_norm = c_enc.abs()
    else:
        l2_norm = c_enc.norm(p=2,dim=dim,keepdim=True)
    crypten.print(f"Rank {rank}:\n\tl_2 norm: {l2_norm.get_plain_text()}")
    inv_clip= 1/clip_threshold
    clip = l2_norm.gt(clip_threshold)
    c_enc = (c_enc / (l2_norm*inv_clip) - c_enc) * clip + c_enc
    #crypten.print(f"Clipped vector: {c_enc.get_plain_text()}")
    shape = (*c.shape, 2)  # Two draws of Poisson
    seed = int(time.time() * 10**6)
    #crypten.print(f"Rank {rank}:\n\tseed: {seed}", in_order=True)
    #print(seed)
    torch.manual_seed(seed)
    
    
    poissons = torch.poisson(torch.tensor([poisson_lam, poisson_lam]).repeat(*shape[:-1], 1))
    crypten.print(f"Rank {rank}:\n\tPoissons: {(poissons[..., 0] - poissons[..., 1]).to(torch.int64)}", in_order=True)
    skellam_noise_enc = crypten.cryptensor((poissons[..., 0] - poissons[..., 1]).to(torch.int64), src = rank)
    crypten.print(f"Rank {rank}:\n\tSkellam: {skellam_noise_enc }", in_order=True)
    #print((poissons[..., 0] - poissons[..., 1]).to(v.dtype))
    #crypten.print(f"Rank {rank}:\n\tPre-noise: {c_enc}", in_order=True)
   
    c_enc_post = c_enc + skellam_noise_enc
    crypten.print(f"Rank {rank}:\n\tPost-noise: {c_enc - c_enc_post}", in_order=True)

    crypten.print(c_enc_post.get_plain_text()-c)


c = torch.rand(16)
print(f"Original c value: {c}")
clip_add_noise(c)



Original c value: tensor([4.8284e-01, 1.7135e-01, 3.7976e-01, 1.4702e-01, 9.2872e-01, 5.3060e-04,
        7.5705e-01, 9.7964e-03, 3.0256e-01, 5.4067e-01, 1.9394e-01, 1.1515e-02,
        8.9736e-01, 5.8264e-01, 3.9909e-01, 2.2734e-01])
Rank 0:
	l_2 norm: tensor([4.8283e-01, 1.7134e-01, 3.7976e-01, 1.4702e-01, 9.2871e-01, 5.1880e-04,
        7.5703e-01, 9.7961e-03, 3.0255e-01, 5.4066e-01, 1.9394e-01, 1.1505e-02,
        8.9735e-01, 5.8264e-01, 3.9908e-01, 2.2733e-01])
Rank 0:
	Poissons: tensor([ 2, -1, -3,  0,  0, -1, -1,  1, -1,  2,  1, -1,  0,  0,  1,  0])
Rank 1:
	Poissons: tensor([ 1,  1, -1,  0,  0, -1,  0,  0, -1,  0,  0,  0,  0,  1,  0,  2])
Rank 0:
	Skellam: MPCTensor(
	_tensor=tensor([-4686356145832430785,   551950335434494802,  -667953586819191225,
        -8146490711259980327, -3316744832448377929,    66008004245950393,
         6101161490719885081, -9032988488129092091,  3221880286636434978,
         -603154548009167068,  6371948384543921444, -1313824152860995685,
         60

[None, None]

# Final code for Skellam sampling and clipping

In [11]:
@mpc.run_multiprocess(world_size=2)
def clip_add_noise(c, poisson_lam=.1, clip_threshold=1.0):
    rank = comm.get().get_rank()
    c_enc = crypten.cryptensor(c)
    # In some layer (e.g. RELU) the output dimensionality is 1, so the gradient clipping and noising needs to be done component by componet, which is an abs 
    dim = 1 % len(c_enc.shape)
    if not dim:
        l2_norm = c_enc.abs()
    else:
        l2_norm = c_enc.norm(p=2,dim=dim,keepdim=True)
    l2_norm = c_enc.norm(p=2,dim=1,keepdim=True )
    inv_clip= 1/clip_threshold
    clip = l2_norm.gt(clip_threshold)
    c_enc = (c_enc / (l2_norm*inv_clip) - c_enc) * clip + c_enc
    #crypten.print(f"Clipped vector: {c_enc.get_plain_text()}")
    shape = (*c.shape, 2)  # Two draws of Poisson
    seed = int(time.time() * 10**6)
    #crypten.print(f"Rank {rank}:\n\tseed: {seed}", in_order=True)
    #print(seed)
    torch.manual_seed(seed)
    
    
    poissons = torch.poisson(torch.tensor([poisson_lam, poisson_lam]).repeat(*shape[:-1], 1))
    #crypten.print(f"Rank {rank}:\n\tPoissons: {(poissons[..., 0] - poissons[..., 1]).to(torch.int64)}", in_order=True)
    skellam_noise_enc = crypten.cryptensor((poissons[..., 0] - poissons[..., 1]).to(torch.int64), src = rank)
    #crypten.print(f"Rank {rank}:\n\tSkellam: {skellam_noise_enc }", in_order=True)
    #print((poissons[..., 0] - poissons[..., 1]).to(v.dtype))
    #crypten.print(f"Rank {rank}:\n\tPre-noise: {c_enc}", in_order=True)
   
    c_enc = c_enc + skellam_noise_enc
    #crypten.print(f"Rank {rank}:\n\tPost-noise: {c_enc - c_enc_post}", in_order=True)

    crypten.print(c_enc.get_plain_text())


c = torch.rand(2, 1, 2, 2)
print(f"Original c value: {c}")
clip_add_noise(c)



Original c value: tensor([[[[0.1285, 0.0612],
          [0.9537, 0.4312]]],


        [[[0.5875, 0.8076],
          [0.1184, 0.9700]]]])
tensor([[[[-0.8715,  0.0612],
          [-1.0463, -0.5688]]],


        [[[ 0.5875,  0.8076],
          [ 0.1183,  0.9700]]]])


[None, None]

In [15]:
a = torch.tensor([1,2,3])
b = crypten.cryptensor(a, src=0, requires_grad = True)
#c = crypten.cryptensor(b, requires_grad = True)